# LeetCode #480: Sliding Window Median

https://leetcode.com/problems/sliding-window-median/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Sort Each Window)** | $O(n \cdot k \log k)$ | $O(k)$ |
| **Optimal: Two Heaps with Lazy Deletion ★** | $O(n \log k)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force (Sort Each Window)
For each window position, sort the k elements and pick the median. Straightforward but slow.

### Optimal: Two Heaps with Lazy Deletion ★
Maintain a max-heap for the lower half and a min-heap for the upper half, balanced so the median is at the top of one or both heaps. When the window slides, add the new element and mark the outgoing element for lazy deletion. Rebalance the heaps after each operation.

**Why this is better than Brute Force:** Heap operations are O(log k) per slide, giving O(n log k) total vs O(nk log k) for re-sorting each window.

**Constraints:**
* 1 <= k <= nums.length <= 10^5
* -2^31 <= nums[i] <= 2^31 - 1

## Solutions

### C#

In [ ]:
public class Solution {
    public double[] MedianSlidingWindow(int[] nums, int k) {
        var small = new SortedList<(int val, int idx), int>();
        var large = new SortedList<(int val, int idx), int>();
        int n = nums.Length;
        double[] result = new double[n - k + 1];

        for (int i = 0; i < n; i++) {
            // Add to large first, then move smallest of large to small
            large[(nums[i], i)] = 0;
            var move = large.Keys[0];
            large.Remove(move);
            small[move] = 0;
            // Balance: small can have at most one more than large
            if (small.Count > large.Count + 1) {
                var mv = small.Keys[small.Count - 1];
                small.Remove(mv);
                large[mv] = 0;
            }
            if (i >= k - 1) {
                if (k % 2 == 1)
                    result[i - k + 1] = small.Keys[small.Count - 1].val;
                else
                    result[i - k + 1] = ((double)small.Keys[small.Count - 1].val + large.Keys[0].val) / 2;
                // Remove outgoing element
                var rem = (nums[i - k + 1], i - k + 1);
                if (small.ContainsKey(rem)) small.Remove(rem);
                else large.Remove(rem);
                // Rebalance
                if (small.Count < large.Count) {
                    var mv = large.Keys[0];
                    large.Remove(mv);
                    small[mv] = 0;
                } else if (small.Count > large.Count + 1) {
                    var mv = small.Keys[small.Count - 1];
                    small.Remove(mv);
                    large[mv] = 0;
                }
            }
        }
        return result;
    }
}

### Python

In [ ]:
from sortedcontainers import SortedList

class Solution:
    def medianSlidingWindow(self, nums: list[int], k: int) -> list[float]:
        sl = SortedList()
        result = []
        for i, v in enumerate(nums):
            sl.add(v)
            if len(sl) > k:
                sl.remove(nums[i - k])
            if len(sl) == k:
                if k % 2 == 1:
                    result.append(float(sl[k // 2]))
                else:
                    result.append((sl[k // 2 - 1] + sl[k // 2]) / 2.0)
        return result

### Go

In [ ]:
import "sort"

func medianSlidingWindow(nums []int, k int) []float64 {
    // Using a sorted slice approach
    window := make([]int, 0, k)
    result := make([]float64, 0, len(nums)-k+1)

    for i, v := range nums {
        pos := sort.SearchInts(window, v)
        window = append(window, 0)
        copy(window[pos+1:], window[pos:])
        window[pos] = v

        if len(window) > k {
            old := nums[i-k]
            p := sort.SearchInts(window, old)
            window = append(window[:p], window[p+1:]...)
        }
        if len(window) == k {
            if k%2 == 1 {
                result = append(result, float64(window[k/2]))
            } else {
                result = append(result, (float64(window[k/2-1])+float64(window[k/2]))/2.0)
            }
        }
    }
    return result
}

### Rust

In [ ]:
use std::collections::BTreeMap;

impl Solution {
    pub fn median_sliding_window(nums: Vec<i32>, k: i32) -> Vec<f64> {
        let k = k as usize;
        let mut window: Vec<i32> = Vec::with_capacity(k);
        let mut result = Vec::new();
        for (i, &v) in nums.iter().enumerate() {
            let pos = window.binary_search(&v).unwrap_or_else(|x| x);
            window.insert(pos, v);
            if window.len() > k {
                let old = nums[i - k];
                let p = window.binary_search(&old).unwrap();
                window.remove(p);
            }
            if window.len() == k {
                if k % 2 == 1 {
                    result.push(window[k / 2] as f64);
                } else {
                    result.push((window[k / 2 - 1] as f64 + window[k / 2] as f64) / 2.0);
                }
            }
        }
        result
    }
}

## Example Scenarios

### Scenario 1: Odd window size
**Input:** `nums = [1,3,-1,-3,5,3,6,7], k = 3`  
Windows: [1,3,-1]->1, [3,-1,-3]->-1, [-1,-3,5]->-1, [-3,5,3]->3, [5,3,6]->5, [3,6,7]->6. **Output:** `[1.0,-1.0,-1.0,3.0,5.0,6.0]`

### Scenario 2: Even window size
**Input:** `nums = [1,2,3,4], k = 2`  
Medians: (1+2)/2=1.5, (2+3)/2=2.5, (3+4)/2=3.5. **Output:** `[1.5,2.5,3.5]`

### Scenario 3: Window equals array
**Input:** `nums = [5,2,8], k = 3`  
Sorted: [2,5,8], median is 5. **Output:** `[5.0]`

### Scenario 4: All same elements
**Input:** `nums = [7,7,7,7], k = 2`  
Every window has median 7. **Output:** `[7.0,7.0,7.0]`

### Scenario 5: Integer overflow concern
**Input:** `nums = [2147483647,2147483647], k = 2`  
Median = (2147483647 + 2147483647) / 2.0 = 2147483647.0. Must use floating point to avoid overflow. **Output:** `[2147483647.0]`

![image](attachment:image.png)